# Clinical-LLM — QLoRA fine-tune on a Colab GPU

End-to-end: install → build a medical instruction corpus → **QLoRA fine-tune** 
`Qwen2.5-3B-Instruct` → evaluate base vs. fine-tuned on medical-QA benchmarks → 
chat with the result.

> ⚠️ **Research/education only — not medical advice, not a medical device.** 
> Do not enter real patient data (PHI).

**Before you run anything:** switch to a GPU runtime — 
**Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is free and works).

## 0. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU'
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability()
native_bf16 = major >= 8
print(f'CUDA device: {name}  (compute capability {major}.{minor})')
print(f'native bf16 : {native_bf16}')
print(f'FlashAttn-2 : {"supported" if native_bf16 else "NOT supported on this GPU"}')
print()
if native_bf16:
    print('-> use configs/train_gpu.yaml   (~45-70 min on an L4, ~20-30 min on an A100)')
else:
    print('-> use configs/train_colab.yaml (T4 class: fp16 fallback, ~2-3 h)')
    print('   A T4 has no native bf16 and no FlashAttention-2. It works, just slowly.')


## 1. Get the code

Run the cell below. If it can't find `clinical-llm.zip` on the runtime it opens an
upload picker — choose the zip and it continues.

**Re-running this cell after uploading a newer zip refreshes the code in place.**
It extracts over the existing folder rather than reusing it, so a patched zip
actually takes effect; `outputs/` and `data/` are left alone, so a trained
adapter or a built corpus survives the refresh. Set `CLEAN = True` for a
scorched-earth re-extract.


In [ ]:
import glob, os, shutil, zipfile

TARGET = "/content/clinical-llm"
MARKER = "pyproject.toml"
CLEAN  = False      # True = delete TARGET first (also wipes outputs/ and data/)


def _find_zip():
    pats = ["/content/clinical-llm*.zip", "/content/*.zip",
            "/content/**/clinical-llm*.zip", os.path.join(os.getcwd(), "*.zip")]
    hits = []
    for pat in pats:
        hits += glob.glob(pat, recursive=True)
    hits = [h for h in dict.fromkeys(hits) if not h.startswith(TARGET + os.sep)]
    return sorted(hits, key=os.path.getmtime, reverse=True)   # newest first


def _extract(src):
    print(f"unpacking {src} ({os.path.getsize(src)/1024:.0f} KB)")
    if CLEAN:
        shutil.rmtree(TARGET, ignore_errors=True)
    with zipfile.ZipFile(src) as z:
        tops = {n.replace("./", "", 1).split("/")[0] for n in z.namelist() if n.strip("./")}
        # Extract OVER the target: code files are overwritten, outputs/ and
        # data/ are untouched. Reusing a stale folder is how a patched zip
        # silently fails to take effect.
        z.extractall("/content" if tops == {"clinical-llm"} else TARGET)


zips = _find_zip()
if zips:
    _extract(zips[0])
elif not os.path.exists(os.path.join(TARGET, MARKER)):
    print("No clinical-llm*.zip found. /content currently holds:")
    print("   ", sorted(os.listdir("/content")) or "(empty)")
    print("\nOpening an upload picker - select clinical-llm.zip.\n")
    from google.colab import files

    uploaded = files.upload()
    zips = [os.path.join("/content", n) for n in uploaded if n.endswith(".zip")]
    if not zips:
        raise SystemExit("No .zip uploaded. Re-run this cell and pick clinical-llm.zip.")
    _extract(zips[0])
else:
    print(f"No zip found, but {TARGET} exists - using what is already there.")

os.chdir(TARGET)
if not os.path.exists(MARKER):
    raise SystemExit(
        f"{TARGET} has no {MARKER}. Contents: {sorted(os.listdir(TARGET))[:20]}"
    )

# Which code is actually live? Cheap fingerprint so a stale runtime is obvious.
have_gpu_cfg = os.path.exists("configs/train_gpu.yaml")
have_guard = "validate_adapter" in open("src/clinical_llm/eval/run_eval.py").read()
have_prec = "resolve_precision" in open("src/clinical_llm/train/train_qlora.py").read()
print("\ncwd:", os.getcwd())
print(f"  train_gpu.yaml      : {'yes' if have_gpu_cfg else 'NO  <- stale code'}")
print(f"  eval adapter guard  : {'yes' if have_guard else 'NO  <- stale code'}")
print(f"  precision auto-fix  : {'yes' if have_prec else 'NO  <- stale code'}")
if not (have_gpu_cfg and have_guard and have_prec):
    print("\n!! This runtime has OLD code. Upload the latest clinical-llm.zip and re-run.")


## 2. Install dependencies

Installs torch/transformers/trl/peft + **bitsandbytes** (the 4-bit backend that 
makes QLoRA possible — CUDA only, which is why this needs the GPU runtime).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
# quick sanity check that the 4-bit backend is present
import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__)

## 3. Build the instruction corpus

Corpus size and step count have to move together. Packing merges short QA
samples into full-length sequences (~2.5 samples per 1024 tokens), so a small
corpus plus a large `max_steps` silently becomes many epochs of overfitting.

On an L4/A100 this picks `data_gpu.yaml` (~75k examples) to match
`train_gpu.yaml`'s 500 steps ≈ 1.1 epochs. On a T4 it picks `data_colab.yaml`
(~16.6k) to match `train_colab.yaml`'s 300 steps. Expect the bigger corpus to
take several minutes rather than one.


In [ ]:
import torch

ampere = torch.cuda.get_device_capability()[0] >= 8
DATA_CONFIG = "configs/data_gpu.yaml" if ampere else "configs/data_colab.yaml"
print(f"using {DATA_CONFIG}  (this pairs with "
      f"{'train_gpu.yaml / 500 steps' if ampere else 'train_colab.yaml / 300 steps'})\n")

!python -m clinical_llm.data.prepare --config {DATA_CONFIG}
!wc -l data/processed/train.jsonl data/processed/val.jsonl


In [ ]:
# peek at one formatted example
import json
with open('data/processed/train.jsonl') as f:
    rec = json.loads(f.readline())
for m in rec['messages']:
    print(f"--- {m['role'].upper()} ---\n{m['content'][:300]}\n")

## 4. QLoRA fine-tune

Picks the config from the GPU you actually got: `train_gpu.yaml` on an L4/A100
(native bf16 + FlashAttention-2), `train_colab.yaml` on a T4 (fp16 fallback).

Two lines in the startup log are worth checking before you walk away —
`precision=` should read `bf16` on an L4/A100 and `fp16` on a T4 (**never
`fp32`**), and the attention implementation should read `flash_attention_2`
on an L4/A100. Then multiply the `s/it` from the progress bar by `max_steps`
and confirm the total fits your session.


In [ ]:
import torch, yaml

ampere = torch.cuda.get_device_capability()[0] >= 8
CONFIG = "configs/train_gpu.yaml" if ampere else "configs/train_colab.yaml"

# Sanity-check the step budget against the corpus that actually got built,
# rather than trusting that the right data config was used.
cfg = yaml.safe_load(open(CONFIG))
n_train = sum(1 for _ in open("data/processed/train.jsonl"))
eff = cfg["per_device_batch_size"] * cfg["grad_accum"]
packed = n_train / (cfg["model"]["max_seq_len"] / 1024 * 2.5)
epochs = cfg["max_steps"] / (packed / eff)
print(f"config     : {CONFIG}")
print(f"train rows : {n_train:,}")
print(f"eff. batch : {eff}   seq-len {cfg['model']['max_seq_len']}")
print(f"max_steps  : {cfg['max_steps']}  ->  ~{epochs:.2f} epochs")
if epochs > 2.5:
    print("\n!! That is a lot of epochs for a LoRA -- you likely built the small")
    print("   corpus. Re-run the corpus cell, or lower max_steps.")
print()

!python -m clinical_llm.train.train_qlora --config {CONFIG}


## 5. Evaluate — base vs. fine-tuned

Scores accuracy by letter-constrained log-likelihood on held-out MedMCQA + PubMedQA. 
`--max-items 300` keeps eval quick; raise for tighter estimates. The printed 
`delta` is your headline number for the model card.

In [ ]:
import glob, json, os, subprocess, sys

ADAPTER = "outputs/clinical-qlora"

# Eval is meaningless without a trained adapter, and PEFT's failure mode for a
# missing local path is a confusing Hub 404 plus "Invalid username or password".
# Plain if/else, not raise -- a guard firing is normal, not an exception.
if not os.path.exists(os.path.join(ADAPTER, "adapter_config.json")):
    ckpts = sorted(glob.glob(os.path.join(ADAPTER, "checkpoint-*")),
                   key=lambda c: int(c.rsplit("-", 1)[-1]))
    print(f"Nothing to evaluate yet: no adapter_config.json in {ADAPTER}/\n")
    print("  outputs/    :", sorted(os.listdir("outputs")) if os.path.isdir("outputs") else "(missing)")
    print("  checkpoints :", [os.path.basename(c) for c in ckpts] or "none")
    print()
    if ckpts:
        print(f"Training ran but stopped before the final save. Score the newest")
        print(f"checkpoint by setting ADAPTER = {ckpts[-1]!r} above and re-running.")
    elif os.path.isdir(ADAPTER):
        print("The output dir exists but is empty -- the trainer creates it at startup,")
        print("so this is what an interrupted run leaves behind. Re-run the fine-tune")
        print("cell and let it reach at least the first checkpoint.")
    else:
        print("Run the fine-tune cell first.")
else:
    rc = subprocess.call([
        sys.executable, "-m", "clinical_llm.eval.run_eval",
        "--base", "Qwen/Qwen2.5-3B-Instruct",
        "--adapter", ADAPTER,
        "--benchmarks", "medmcqa", "pubmedqa",
        "--max-items", "300",
        "--out", "reports/eval.json",
    ])
    if rc != 0:
        print(f"\nrun_eval exited {rc} -- see the traceback above")
    else:
        deltas = json.load(open("reports/eval.json")).get("deltas", {})
        print("\n=== base vs fine-tuned ===")
        print(json.dumps(deltas, indent=2))


## 6. Chat with the fine-tuned model

In [ ]:
from clinical_llm.serve.inference import ClinicalLLM, GenerationSettings
llm = ClinicalLLM(base_model='Qwen/Qwen2.5-3B-Instruct', adapter='outputs/clinical-qlora')

for q in [
    'What is the first-line treatment for anaphylaxis, and by what route?',
    'Explain the difference between type 1 and type 2 respiratory failure.',
]:
    print('Q:', q)
    print('A:', llm.generate(q, settings=GenerationSettings(max_new_tokens=200, temperature=0.3)))
    print('-' * 80)

## 7. Save your adapter

The LoRA adapter is tiny (tens of MB). Download it, or mount Google Drive to keep it.

In [ ]:
# Option A: download a zip of the adapter
!cd outputs && zip -qr clinical-qlora.zip clinical-qlora
from google.colab import files
files.download('outputs/clinical-qlora.zip')

In [ ]:
# Option B: save to Google Drive instead
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r outputs/clinical-qlora /content/drive/MyDrive/

---
Next: paste the eval deltas into `docs/MODEL_CARD.md`, and serve the adapter locally 
with `CLINICAL_LLM_ADAPTER=outputs/clinical-qlora uvicorn clinical_llm.serve.app:app`.